# UNSW-NB15 — End-to-End Network Traffic Clustering

**Goal:** Build an unsupervised clustering workflow for network traffic behavior.

The notebook starts with EDA, then continues through:

**Cleaning → Feature Selection → Encoding / Log Transformation → Scaling → PCA → KMeans / DBSCAN / Agglomerative → Evaluation → Cluster Profiling → External Label Check → Saving / Deployment Setup**

> `attack_label` and `attack_category` are **reference-only columns**. They are never used as clustering input features.


In [ ]:
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.options.display.float_format = '{:,.3f}'.format

sns.set_theme(style='whitegrid')

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.cluster import (
    KMeans,
    DBSCAN,
    AgglomerativeClustering,
)
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")

import sklearn
print(f"scikit-learn : {sklearn.__version__}")

## 1. Load Dataset


In [ ]:
FILENAME = "UNSW_NB15_training-set.parquet"

possible_paths = [
    Path(FILENAME),
    Path("data") / FILENAME,
    Path("/content") / FILENAME,
    Path("/content/sample_data") / FILENAME,
    Path("/content/drive/MyDrive") / FILENAME,
]

data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    try:
        from google.colab import files
        print("Dataset not found. Choose the Parquet file to upload:")
        uploaded = files.upload()
        data_path = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError(
            "Dataset not found. Place it beside the notebook or inside a data folder."
        )

df = pd.read_parquet(data_path)

print("Loaded from:", data_path)
print("Shape:", df.shape)
display(df.head())

## 2. Basic Inspection


In [ ]:
print("Rows   :", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

### Dataset Structure

In [ ]:
df.info()

### Analysis

In [ ]:
display(df.head(10))

### Analysis

In [ ]:
display(df.sample(10, random_state=42))

## 2.1 Rename Abbreviated Columns

This changes **column names only**. It does not change any data values.


In [ ]:
RENAME_MAP = {
    'state': 'connection_state',
    'attack_cat': 'attack_category',
    'service': 'network_service',
    'proto': 'protocol',

    'ackdat': 'acknowledgement_time',
    'synack': 'syn_to_ack_time',
    'tcprtt': 'tcp_round_trip_time',
    'dur': 'duration',

    'dinpkt': 'destination_interpacket_time',
    'sinpkt': 'source_interpacket_time',

    'djit': 'destination_jitter',
    'sjit': 'source_jitter',

    'rate': 'transmission_rate',
    'dload': 'destination_load',
    'sload': 'source_load',

    'dwin': 'destination_tcp_window',
    'swin': 'source_tcp_window',

    'trans_depth': 'transaction_depth',

    'dloss': 'destination_packet_loss',
    'sloss': 'source_packet_loss',

    'dpkts': 'destination_packets',
    'spkts': 'source_packets',

    'dmean': 'destination_mean_packet_size',
    'smean': 'source_mean_packet_size',

    'response_body_len': 'response_body_length',

    'dbytes': 'destination_bytes',
    'sbytes': 'source_bytes',

    'dtcpb': 'destination_tcp_base_sequence',
    'stcpb': 'source_tcp_base_sequence',

    'is_sm_ips_ports': 'same_source_destination_ip_port',
    'label': 'attack_label',

    'is_ftp_login': 'ftp_login_status',
    'ct_ftp_cmd': 'ftp_command_count',

    'ct_flw_http_mthd': 'http_method_flow_count',

    'ct_dst_sport_ltm': 'same_destination_source_port_count',
    'ct_src_dport_ltm': 'same_source_destination_port_count',
}

df = df.rename(columns=RENAME_MAP)

print("Renaming complete.")
print(df.columns.tolist())

## 3. Data Types


In [ ]:
dtype_table = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(dropna=False),
    'missing': df.isna().sum()
}).sort_values(['dtype', 'n_unique'])

display(dtype_table)

### Results

In [ ]:
categorical_cols = df.select_dtypes(
    include=['object', 'category', 'string']
).columns.tolist()

numerical_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

                                           
                                                 
reference_cols = [
    c for c in ['attack_label', 'attack_category']
    if c in df.columns
]

clustering_numerical_cols = [
    c for c in numerical_cols
    if c not in reference_cols
]

clustering_categorical_cols = [
    c for c in categorical_cols
    if c not in reference_cols
]

print("Categorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numerical_cols)

print("\nReference-only columns:")
print(reference_cols)

## 4. Missing Values


In [ ]:
missing_table = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(3)
}).sort_values('missing_count', ascending=False)

display(missing_table)

print("Total missing cells:", int(df.isna().sum().sum()))

## 5. Duplicate Rows


In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_percent = duplicate_count / len(df) * 100

print(f"Exact duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percent:.3f}%")

### Results

In [ ]:
                                                
if df.duplicated(keep=False).any():
    display(
        df[df.duplicated(keep=False)]
        .sort_values(by=df.columns.tolist())
        .head(30)
    )
else:
    print("No exact duplicate rows found.")

## 6. Infinite Values


In [ ]:
numeric_df = df.select_dtypes(include=np.number)

inf_counts = pd.Series(
    np.isinf(numeric_df).sum(axis=0),
    index=numeric_df.columns,
    name='infinite_count'
)

display(inf_counts[inf_counts > 0].sort_values(ascending=False))

if (inf_counts > 0).sum() == 0:
    print("No +inf / -inf values found.")

## 7. Cardinality / Unique Values


In [ ]:
cardinality = pd.DataFrame({
    'n_unique': df.nunique(dropna=False),
    'unique_percent': (df.nunique(dropna=False) / len(df) * 100).round(3)
}).sort_values('n_unique')

display(cardinality)

## 8. Constant and Near-Constant Features


In [ ]:
feature_quality = []

for col in df.columns:
    counts = df[col].value_counts(dropna=False, normalize=True)
    top_ratio = counts.iloc[0] * 100 if len(counts) else np.nan

    feature_quality.append({
        'feature': col,
        'n_unique': df[col].nunique(dropna=False),
        'most_frequent_percent': top_ratio
    })

feature_quality_df = pd.DataFrame(feature_quality).sort_values(
    ['n_unique', 'most_frequent_percent'],
    ascending=[True, False]
)

display(feature_quality_df)

print("\nConstant features:")
display(feature_quality_df[feature_quality_df['n_unique'] <= 1])

print("\nNear-constant features (>= 95% one value):")
display(
    feature_quality_df[
        (feature_quality_df['most_frequent_percent'] >= 95)
        & (feature_quality_df['n_unique'] > 1)
    ]
)

## 9. Numerical Descriptive Statistics


In [ ]:
display(
    df[clustering_numerical_cols]
    .describe()
    .T
)

## 10. Zero Values in Numerical Features


In [ ]:
zero_table = pd.DataFrame({
    'zero_count': (df[clustering_numerical_cols] == 0).sum(),
    'zero_percent': (
        (df[clustering_numerical_cols] == 0).mean() * 100
    ).round(2)
}).sort_values('zero_percent', ascending=False)

display(zero_table)

## 11. Skewness


In [ ]:
skewness = (
    df[clustering_numerical_cols]
    .skew(numeric_only=True)
    .sort_values(ascending=False)
)

skewness_df = pd.DataFrame({
    'skewness': skewness,
    'abs_skewness': skewness.abs()
}).sort_values('abs_skewness', ascending=False)

display(skewness_df)

## 12. Numerical Distributions

For plotting only, we use a random sample so the notebook stays responsive.  
All descriptive statistics above were calculated using the full dataset.


In [ ]:
PLOT_SAMPLE_SIZE = min(30_000, len(df))

plot_df = df.sample(
    PLOT_SAMPLE_SIZE,
    random_state=42
)

print("Rows used for plots:", len(plot_df))

### Visual Analysis

In [ ]:
                             

cols_per_figure = 6

for start in range(0, len(clustering_numerical_cols), cols_per_figure):
    cols = clustering_numerical_cols[start:start + cols_per_figure]

    fig, axes = plt.subplots(
        len(cols),
        1,
        figsize=(11, 3.2 * len(cols))
    )

    if len(cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, cols):
        sns.histplot(
            plot_df[col].dropna(),
            bins=40,
            kde=True,
            ax=ax
        )
        ax.set_title(f'Distribution of {col}')
        ax.set_xlabel(col)

    plt.tight_layout()
    plt.show()

## 13. Boxplots / Statistical Outliers


In [ ]:
                            
                                                                            
                                   

cols_per_figure = 6

for start in range(0, len(clustering_numerical_cols), cols_per_figure):
    cols = clustering_numerical_cols[start:start + cols_per_figure]

    fig, axes = plt.subplots(
        len(cols),
        1,
        figsize=(11, 2.7 * len(cols))
    )

    if len(cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, cols):
        sns.boxplot(
            x=plot_df[col],
            ax=ax
        )
        ax.set_title(f'Boxplot of {col}')

    plt.tight_layout()
    plt.show()

## 14. IQR Outlier Count


In [ ]:
iqr_results = []

for col in clustering_numerical_cols:
    series = df[col].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_mask = (series < lower) | (series > upper)

    iqr_results.append({
        'feature': col,
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'lower_bound': lower,
        'upper_bound': upper,
        'outlier_count': outlier_mask.sum(),
        'outlier_percent': outlier_mask.mean() * 100
    })

iqr_outliers_df = (
    pd.DataFrame(iqr_results)
    .sort_values('outlier_percent', ascending=False)
)

display(iqr_outliers_df)

## 15. Optional Log-Scale View for Highly Skewed Non-Negative Features

This does **not** change `df`. It is only a temporary EDA view using `np.log1p()`.


In [ ]:
highly_skewed_cols = skewness_df[
    skewness_df['abs_skewness'] >= 2
].index.tolist()

nonnegative_high_skew = [
    col for col in highly_skewed_cols
    if df[col].min(skipna=True) >= 0
]

print("Highly skewed, non-negative columns:")
print(nonnegative_high_skew)

### Visual Analysis

In [ ]:
for start in range(0, len(nonnegative_high_skew), 6):
    cols = nonnegative_high_skew[start:start + 6]

    fig, axes = plt.subplots(
        len(cols),
        1,
        figsize=(11, 3.2 * len(cols))
    )

    if len(cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, cols):
        sns.histplot(
            np.log1p(plot_df[col].dropna()),
            bins=40,
            kde=True,
            ax=ax
        )
        ax.set_title(f'log1p({col}) — EDA View Only')
        ax.set_xlabel(f'log1p({col})')

    plt.tight_layout()
    plt.show()

## 16. Categorical Feature Distributions


In [ ]:
for col in clustering_categorical_cols:
    n_unique = df[col].nunique(dropna=False)

    print(f"\n{'=' * 70}")
    print(f"{col} | unique values = {n_unique}")
    print('=' * 70)

    display(
        df[col]
        .value_counts(dropna=False)
        .rename('count')
        .to_frame()
        .head(30)
    )

                                                            
    top_n = n_unique if n_unique <= 20 else 20

    order = (
        df[col]
        .value_counts()
        .head(top_n)
        .index
    )

    plt.figure(figsize=(11, max(4, top_n * 0.32)))
    sns.countplot(
        data=plot_df[plot_df[col].isin(order)],
        y=col,
        order=order
    )

    title_suffix = '' if n_unique <= 20 else ' — Top 20'
    plt.title(f'{col} Distribution{title_suffix}')
    plt.tight_layout()
    plt.show()

## 17. Provided Labels — Reference Only

These columns are available in the dataset, but they are **not clustering features**.

We inspect them only so that later, after clustering, we can compare discovered clusters with the known traffic labels.


In [ ]:
if 'attack_label' in df.columns:
    label_counts = df['attack_label'].value_counts(dropna=False).sort_index()

    display(
        pd.DataFrame({
            'count': label_counts,
            'percent': (label_counts / len(df) * 100).round(2)
        })
    )

    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x='attack_label')
    plt.title('Attack Label Distribution — Reference Only')
    plt.show()

### Visual Analysis

In [ ]:
if 'attack_category' in df.columns:
    attack_counts = df['attack_category'].value_counts(dropna=False)

    display(
        pd.DataFrame({
            'count': attack_counts,
            'percent': (attack_counts / len(df) * 100).round(2)
        })
    )

    plt.figure(figsize=(11, 6))
    order = df['attack_category'].value_counts().index
    sns.countplot(data=df, y='attack_category', order=order)
    plt.title('Attack Category Distribution — Reference Only')
    plt.tight_layout()
    plt.show()

## 18. Numerical Correlation Heatmap — Clustering Features Only


In [ ]:
corr = df[clustering_numerical_cols].corr()

plt.figure(figsize=(18, 15))
sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    square=False,
    linewidths=0.2
)

plt.title('Correlation Heatmap — Numerical Clustering Features')
plt.tight_layout()
plt.show()

## 19. Strong Correlation Pairs


In [ ]:
corr_abs = corr.abs()

upper_triangle = corr_abs.where(
    np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
)

strong_corr_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

strong_corr_pairs.columns = [
    'feature_1',
    'feature_2',
    'absolute_correlation'
]

strong_corr_pairs = strong_corr_pairs.sort_values(
    'absolute_correlation',
    ascending=False
)

display(strong_corr_pairs.head(30))

## 20. Categorical Cross-Tabs — Reference Only


In [ ]:
if 'attack_label' in df.columns:
    label_counts = df['attack_label'].value_counts(dropna=False).sort_index()

    display(
        pd.DataFrame({
            'count': label_counts,
            'percent': (label_counts / len(df) * 100).round(2)
        })
    )

    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x='attack_label')
    plt.title('Attack Label Distribution — Reference Only')
    plt.show()

## 21. EDA Summary / Checklist

Before preprocessing and clustering, the EDA established:

- Dataset shape and feature types
- Missing-value status
- Duplicate-row status
- Constant / near-constant features
- Highly skewed numerical features
- Features with many valid zero values
- Statistical outliers
- High-cardinality categorical features such as `protocol`
- Strong numerical correlations / redundancy candidates
- `attack_label` balance
- `attack_category` distribution

### Main preprocessing conclusions

- Do **not** convert every zero to `NaN`; many zeros are meaningful network values.
- Do **not** automatically delete IQR outliers; extreme traffic may contain the behavior we want to discover.
- Strongly right-skewed, non-negative numerical features can be compressed with `log1p`.
- `attack_label` and `attack_category` stay outside the clustering feature matrix.
- Highly redundant features should be reviewed before PCA.

The EDA stage ends here.


# 22. Cleaning

Decisions used here:

1. No imputer is used because the EDA showed no missing values.
2. Valid zeros are kept as zeros.
3. IQR outliers are **not automatically removed**.
4. If an `id` column exists, it is treated as an identifier, not a behavior feature.
5. Duplicate network records are deduplicated for clustering geometry, but their original occurrence count is preserved in `row_frequency` for later profiling.

This avoids giving repeated identical rows artificial extra influence while still preserving how frequent each pattern was in the original traffic.


In [ ]:
                         
work_df = df.copy()

                                                                          
identifier_cols = [c for c in ['id'] if c in work_df.columns]

                                                                 
duplicate_key_cols = [
    c for c in work_df.columns
    if c not in identifier_cols
]

duplicate_count_behavior = work_df.duplicated(
    subset=duplicate_key_cols
).sum()

duplicate_pct_behavior = (
    duplicate_count_behavior / len(work_df) * 100
)

print(f"Rows before cleaning              : {len(work_df):,}")
print(f"Duplicate behavior rows           : {duplicate_count_behavior:,}")
print(f"Duplicate behavior percentage     : {duplicate_pct_behavior:.2f}%")

                                                                         
clean_df = (
    work_df
    .groupby(
        duplicate_key_cols,
        dropna=False,
        observed=True,
        sort=False
    )
    .size()
    .reset_index(name='row_frequency')
)

print(f"Unique network behavior rows      : {len(clean_df):,}")
print(f"Original rows represented         : {clean_df['row_frequency'].sum():,}")

### Results

In [ ]:
                                              

remaining_missing = int(clean_df.isna().sum().sum())

numeric_clean = clean_df.select_dtypes(include=np.number)
remaining_inf = int(np.isinf(numeric_clean).sum().sum())

print("Missing cells after cleaning :", remaining_missing)
print("Infinite values after cleaning:", remaining_inf)

if remaining_missing == 0:
    print("No imputation is required.")

# 23. Feature Selection

We use a **conservative** feature-selection strategy:

- Remove reference labels.
- Remove identifier-like columns.
- Remove constant features.
- Remove TCP base sequence-number features because their absolute magnitude is not a meaningful behavioral distance.
- Drop only one clearly redundant FTP feature by default.
- Keep other highly correlated but semantically different network features; PCA will handle much of the remaining redundancy.

This avoids aggressively deleting potentially useful attack behavior.


In [ ]:
reference_cols = [
    c for c in ['attack_label', 'attack_category']
    if c in clean_df.columns
]

metadata_cols = [
    c for c in ['row_frequency']
    if c in clean_df.columns
]

feature_candidates = [
    c for c in clean_df.columns
    if c not in reference_cols + metadata_cols
]

constant_cols = [
    c for c in feature_candidates
    if clean_df[c].nunique(dropna=False) <= 1
]

                                                                        
                                                                  
non_behavioral_numeric_cols = [
    c for c in [
        'source_tcp_base_sequence',
        'destination_tcp_base_sequence'
    ]
    if c in feature_candidates
]

                                                                       
                                                               
manual_redundant_drop = [
    c for c in ['ftp_command_count']
    if c in feature_candidates
]

feature_drop_cols = sorted(set(
    constant_cols
    + non_behavioral_numeric_cols
    + manual_redundant_drop
))

selected_feature_cols = [
    c for c in feature_candidates
    if c not in feature_drop_cols
]

print("Reference-only columns:")
print(reference_cols)

print("\nConstant columns dropped:")
print(constant_cols)

print("\nNon-behavioral numeric columns dropped:")
print(non_behavioral_numeric_cols)

print("\nRedundant columns dropped:")
print(manual_redundant_drop)

print(f"\nSelected clustering features: {len(selected_feature_cols)}")
print(selected_feature_cols)

### Results

In [ ]:
                                                                        
selected_numeric_for_corr = (
    clean_df[selected_feature_cols]
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

selected_corr = clean_df[selected_numeric_for_corr].corr().abs()

selected_upper = selected_corr.where(
    np.triu(np.ones(selected_corr.shape), k=1).astype(bool)
)

remaining_strong_pairs = (
    selected_upper
    .stack()
    .reset_index()
)

remaining_strong_pairs.columns = [
    'feature_1',
    'feature_2',
    'absolute_correlation'
]

remaining_strong_pairs = (
    remaining_strong_pairs[
        remaining_strong_pairs['absolute_correlation'] >= 0.90
    ]
    .sort_values('absolute_correlation', ascending=False)
)

display(remaining_strong_pairs)

print(
    "\nThese remaining strong relationships are kept intentionally; "
    "PCA will reduce redundant dimensions later."
)

# 24. Separate Feature Types

We use three groups:

- **Categorical nominal:** `OneHotEncoder`
- **Continuous / count numerical:** `log1p` when strongly right-skewed, then `StandardScaler`
- **Binary 0/1 features:** kept as 0/1

`OrdinalEncoder` is not used because protocol/service/state do not have a meaningful order.


In [ ]:
X = clean_df[selected_feature_cols].copy()

categorical_feature_cols = X.select_dtypes(
    include=['object', 'category', 'string']
).columns.tolist()

numeric_feature_cols = X.select_dtypes(
    include=np.number
).columns.tolist()


def is_binary_zero_one(series):
    values = set(pd.Series(series).dropna().unique().tolist())
    return (
        len(values) <= 2
        and values.issubset({0, 1, 0.0, 1.0})
    )


binary_feature_cols = [
    c for c in numeric_feature_cols
    if is_binary_zero_one(X[c])
]

continuous_numeric_cols = [
    c for c in numeric_feature_cols
    if c not in binary_feature_cols
]

numeric_skewness = X[continuous_numeric_cols].skew(numeric_only=True)

skewed_numeric_cols = [
    c for c in continuous_numeric_cols
    if (
        abs(numeric_skewness[c]) >= 2
        and X[c].min(skipna=True) >= 0
    )
]

regular_numeric_cols = [
    c for c in continuous_numeric_cols
    if c not in skewed_numeric_cols
]

print("Categorical:")
print(categorical_feature_cols)

print("\nBinary 0/1:")
print(binary_feature_cols)

print("\nHighly skewed non-negative numerical:")
print(skewed_numeric_cols)

print("\nOther numerical:")
print(regular_numeric_cols)

# 25. Preprocessing — Encoding + Transformation + Scaling

Pipeline logic:

**Categorical → OneHotEncoder**

**Highly right-skewed numerical → log1p → StandardScaler**

**Other numerical → StandardScaler**

**Binary → passthrough (keep 0/1)**

`StandardScaler` standardizes around mean 0 and standard deviation 1; it does **not** map values to 0–1.


In [ ]:
                                                                
try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse=False
    )

skewed_numeric_pipeline = Pipeline(steps=[
    (
        'log1p',
        FunctionTransformer(
            np.log1p,
            validate=False,
            feature_names_out='one-to-one'
        )
    ),
    ('scaler', StandardScaler()),
])

regular_numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('encoder', one_hot_encoder),
])

transformers = []

if skewed_numeric_cols:
    transformers.append(
        ('skewed_num', skewed_numeric_pipeline, skewed_numeric_cols)
    )

if regular_numeric_cols:
    transformers.append(
        ('regular_num', regular_numeric_pipeline, regular_numeric_cols)
    )

if categorical_feature_cols:
    transformers.append(
        ('categorical', categorical_pipeline, categorical_feature_cols)
    )

if binary_feature_cols:
    transformers.append(
        ('binary', 'passthrough', binary_feature_cols)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder='drop',
    verbose_feature_names_out=False
)

X_preprocessed = preprocessor.fit_transform(X)

                                                        
X_preprocessed = np.asarray(
    X_preprocessed,
    dtype=np.float32
)

print("Original selected feature shape :", X.shape)
print("After preprocessing shape        :", X_preprocessed.shape)
print("All finite values                :", np.isfinite(X_preprocessed).all())

# 26. PCA — Dimensionality Reduction

PCA does **not** create clusters.

It converts the processed feature space into fewer new features:

`PC1, PC2, PC3, ...`

Each principal component is a weighted combination of the processed features.

We keep enough components to explain approximately **95% of the variance**.


In [ ]:
RANDOM_STATE = 42
PCA_VARIANCE_TO_KEEP = 0.95

                                                                     
                                        
PCA_FIT_SAMPLE_SIZE = min(40_000, len(X_preprocessed))

rng = np.random.default_rng(RANDOM_STATE)

if PCA_FIT_SAMPLE_SIZE < len(X_preprocessed):
    pca_fit_idx = rng.choice(
        len(X_preprocessed),
        size=PCA_FIT_SAMPLE_SIZE,
        replace=False
    )
    X_pca_fit = X_preprocessed[pca_fit_idx]
else:
    X_pca_fit = X_preprocessed

pca = PCA(
    n_components=PCA_VARIANCE_TO_KEEP,
    svd_solver='full'
)

pca.fit(X_pca_fit)

X_pca = pca.transform(X_preprocessed).astype(np.float32)

print("Preprocessed dimensions :", X_preprocessed.shape[1])
print("PCA components retained :", pca.n_components_)
print(
    "Explained variance kept :",
    f"{pca.explained_variance_ratio_.sum() * 100:.2f}%"
)
print("PCA data shape          :", X_pca.shape)

### Visual Analysis

In [ ]:
cumulative_variance = np.cumsum(
    pca.explained_variance_ratio_
)

plt.figure(figsize=(9, 5))
plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker='o'
)
plt.axhline(
    PCA_VARIANCE_TO_KEEP,
    linestyle='--'
)
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Cumulative Explained Variance')
plt.tight_layout()
plt.show()

# 27. Practical Clustering Samples

KMeans scales well, but DBSCAN and especially Agglomerative Clustering can become expensive on very large datasets.

We therefore:

- search/tune clustering behavior on a reproducible sample,
- compare final candidate solutions on the **same common sample**,
- then refit on more data when the selected algorithm allows it.

This is a scalability decision, not a train/test split.


In [ ]:
SEARCH_SAMPLE_SIZE = min(5_000, len(X_pca))
COMPARE_SAMPLE_SIZE = min(3_000, len(X_pca))

search_idx = rng.choice(
    len(X_pca),
    size=SEARCH_SAMPLE_SIZE,
    replace=False
)

compare_idx = rng.choice(
    len(X_pca),
    size=COMPARE_SAMPLE_SIZE,
    replace=False
)

X_search = X_pca[search_idx]
X_compare = X_pca[compare_idx]

print("Search sample :", X_search.shape)
print("Compare sample:", X_compare.shape)

### Clustering Evaluation

In [ ]:
def evaluate_clustering(
    X_values,
    labels,
    ignore_noise=False
):
    labels = np.asarray(labels)

    if ignore_noise:
        keep_mask = labels != -1
        X_eval = X_values[keep_mask]
        labels_eval = labels[keep_mask]
        coverage = keep_mask.mean() * 100
    else:
        X_eval = X_values
        labels_eval = labels
        coverage = 100.0

    unique_labels = np.unique(labels_eval)

    if len(unique_labels) < 2:
        return {
            'n_clusters': len(unique_labels),
            'coverage_%': coverage,
            'silhouette': np.nan,
            'davies_bouldin': np.nan,
            'calinski_harabasz': np.nan,
        }

    return {
        'n_clusters': len(unique_labels),
        'coverage_%': coverage,
        'silhouette': silhouette_score(
            X_eval,
            labels_eval
        ),
        'davies_bouldin': davies_bouldin_score(
            X_eval,
            labels_eval
        ),
        'calinski_harabasz': calinski_harabasz_score(
            X_eval,
            labels_eval
        ),
    }

# 28. KMeans

For KMeans:

- `k` = number of clusters
- Elbow Method helps inspect inertia
- Silhouette ↑
- Davies-Bouldin ↓
- Calinski-Harabasz ↑

The code uses Silhouette as the primary automatic choice while still displaying all metrics.


In [ ]:
k_values = range(2, 11)
kmeans_results = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )

    labels = model.fit_predict(X_search)

    scores = evaluate_clustering(
        X_search,
        labels
    )

    kmeans_results.append({
        'k': k,
        'inertia': model.inertia_,
        **scores
    })

kmeans_results_df = pd.DataFrame(kmeans_results)

display(kmeans_results_df)

### Visual Analysis

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    kmeans_results_df['k'],
    kmeans_results_df['inertia'],
    marker='o'
)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('KMeans — Elbow Method')
plt.tight_layout()
plt.show()

### Visual Analysis

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    kmeans_results_df['k'],
    kmeans_results_df['silhouette'],
    marker='o'
)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('KMeans — Silhouette by k')
plt.tight_layout()
plt.show()

best_k_row = (
    kmeans_results_df
    .sort_values(
        ['silhouette', 'davies_bouldin'],
        ascending=[False, True]
    )
    .iloc[0]
)

BEST_K = int(best_k_row['k'])

print("Best KMeans k by Silhouette:", BEST_K)

# 29. DBSCAN

DBSCAN does **not** use `k`.

Important parameters:

- `eps`: neighborhood radius
- `min_samples`: minimum local density needed to form a core region
- label `-1`: noise

A k-distance calculation is used to generate sensible `eps` candidates instead of guessing random values.


In [ ]:
DBSCAN_MIN_SAMPLES_CANDIDATES = [5, 10, 20]

dbscan_results = []

for min_samples in DBSCAN_MIN_SAMPLES_CANDIDATES:

    neighbors = NearestNeighbors(
        n_neighbors=min_samples
    )

    neighbors.fit(X_search)

    distances, _ = neighbors.kneighbors(X_search)

    k_distances = np.sort(
        distances[:, -1]
    )

    positive_distances = k_distances[
        k_distances > 0
    ]

    if len(positive_distances) == 0:
        continue

                                                            
    eps_candidates = np.unique(
        np.quantile(
            positive_distances,
            [0.85, 0.90, 0.95, 0.97, 0.99]
        )
    )

    for eps in eps_candidates:

        model = DBSCAN(
            eps=float(eps),
            min_samples=min_samples
        )

        labels = model.fit_predict(X_search)

        scores = evaluate_clustering(
            X_search,
            labels,
            ignore_noise=True
        )

        noise_pct = (
            (labels == -1).mean() * 100
        )

        dbscan_results.append({
            'eps': float(eps),
            'min_samples': min_samples,
            'noise_%': noise_pct,
            **scores
        })

dbscan_results_df = pd.DataFrame(dbscan_results)

valid_dbscan = dbscan_results_df[
    dbscan_results_df['n_clusters'] >= 2
].copy()

display(
    valid_dbscan
    .sort_values(
        ['silhouette', 'noise_%'],
        ascending=[False, True]
    )
    .head(20)
)

### Results

In [ ]:
                             
                                                                                 

dbscan_eligible = valid_dbscan[
    valid_dbscan['coverage_%'] >= 60
].copy()

if dbscan_eligible.empty:
    dbscan_eligible = valid_dbscan.copy()

if not dbscan_eligible.empty:
    best_dbscan_row = (
        dbscan_eligible
        .sort_values(
            ['silhouette', 'davies_bouldin'],
            ascending=[False, True]
        )
        .iloc[0]
    )

    BEST_DBSCAN_EPS = float(
        best_dbscan_row['eps']
    )

    BEST_DBSCAN_MIN_SAMPLES = int(
        best_dbscan_row['min_samples']
    )

    print("Best DBSCAN eps        :", BEST_DBSCAN_EPS)
    print("Best DBSCAN min_samples:", BEST_DBSCAN_MIN_SAMPLES)
    print(
        "Coverage               :",
        f"{best_dbscan_row['coverage_%']:.2f}%"
    )
else:
    BEST_DBSCAN_EPS = None
    BEST_DBSCAN_MIN_SAMPLES = None
    print("No valid DBSCAN configuration found.")

# 30. Agglomerative Clustering

Agglomerative Clustering is hierarchical:

- it starts with separate points/groups,
- repeatedly merges the closest groups,
- stops at the requested `n_clusters`.

We compare a few values around the strongest KMeans solution and test:

- `ward`
- `complete`
- `average`

This stage is intentionally limited because hierarchical clustering is computationally expensive on large datasets.


In [ ]:
agg_cluster_candidates = sorted(set([
    2,
    3,
    max(2, BEST_K - 1),
    BEST_K,
    BEST_K + 1
]))

agg_linkages = [
    'ward',
    'complete',
    'average'
]

agg_results = []

for linkage in agg_linkages:
    for n_clusters in agg_cluster_candidates:

        model = AgglomerativeClustering(
            n_clusters=n_clusters,
            linkage=linkage
        )

        labels = model.fit_predict(X_compare)

        scores = evaluate_clustering(
            X_compare,
            labels
        )

        agg_results.append({
            'n_clusters_requested': n_clusters,
            'linkage': linkage,
            **scores
        })

agg_results_df = pd.DataFrame(agg_results)

display(
    agg_results_df
    .sort_values(
        ['silhouette', 'davies_bouldin'],
        ascending=[False, True]
    )
)

best_agg_row = (
    agg_results_df
    .sort_values(
        ['silhouette', 'davies_bouldin'],
        ascending=[False, True]
    )
    .iloc[0]
)

BEST_AGG_CLUSTERS = int(
    best_agg_row['n_clusters_requested']
)

BEST_AGG_LINKAGE = str(
    best_agg_row['linkage']
)

print("Best Agglomerative n_clusters:", BEST_AGG_CLUSTERS)
print("Best Agglomerative linkage   :", BEST_AGG_LINKAGE)

# 31. Fair Final Comparison on the Same Sample

All three candidate algorithms are now run on the **same PCA sample**.

Selection direction:

- Silhouette ↑
- Davies-Bouldin ↓
- Calinski-Harabasz ↑

For DBSCAN, `coverage_%` also matters because a very high score is not useful if almost everything was labeled noise.


In [ ]:
comparison_rows = []
comparison_labels = {}
comparison_models = {}

                  
kmeans_compare = KMeans(
    n_clusters=BEST_K,
    random_state=RANDOM_STATE,
    n_init=10
)

kmeans_labels = kmeans_compare.fit_predict(
    X_compare
)

kmeans_scores = evaluate_clustering(
    X_compare,
    kmeans_labels
)

comparison_rows.append({
    'algorithm': 'KMeans',
    'configuration': f'k={BEST_K}',
    **kmeans_scores
})

comparison_labels['KMeans'] = kmeans_labels
comparison_models['KMeans'] = kmeans_compare


                  
if BEST_DBSCAN_EPS is not None:

    dbscan_compare = DBSCAN(
        eps=BEST_DBSCAN_EPS,
        min_samples=BEST_DBSCAN_MIN_SAMPLES
    )

    dbscan_labels = dbscan_compare.fit_predict(
        X_compare
    )

    dbscan_scores = evaluate_clustering(
        X_compare,
        dbscan_labels,
        ignore_noise=True
    )

    comparison_rows.append({
        'algorithm': 'DBSCAN',
        'configuration': (
            f'eps={BEST_DBSCAN_EPS:.4f}, '
            f'min_samples={BEST_DBSCAN_MIN_SAMPLES}'
        ),
        **dbscan_scores
    })

    comparison_labels['DBSCAN'] = dbscan_labels
    comparison_models['DBSCAN'] = dbscan_compare


                         
agg_compare = AgglomerativeClustering(
    n_clusters=BEST_AGG_CLUSTERS,
    linkage=BEST_AGG_LINKAGE
)

agg_labels = agg_compare.fit_predict(
    X_compare
)

agg_scores = evaluate_clustering(
    X_compare,
    agg_labels
)

comparison_rows.append({
    'algorithm': 'Agglomerative',
    'configuration': (
        f'n_clusters={BEST_AGG_CLUSTERS}, '
        f'linkage={BEST_AGG_LINKAGE}'
    ),
    **agg_scores
})

comparison_labels['Agglomerative'] = agg_labels
comparison_models['Agglomerative'] = agg_compare


comparison_df = pd.DataFrame(comparison_rows)

display(
    comparison_df.sort_values(
        ['silhouette', 'davies_bouldin'],
        ascending=[False, True]
    )
)

### Results

In [ ]:
                                 
                                             
                                                                         

eligible_comparison = comparison_df[
    comparison_df['silhouette'].notna()
].copy()

eligible_comparison = eligible_comparison[
    (eligible_comparison['algorithm'] != 'DBSCAN')
    | (eligible_comparison['coverage_%'] >= 60)
]

best_solution_row = (
    eligible_comparison
    .sort_values(
        [
            'silhouette',
            'davies_bouldin',
            'calinski_harabasz'
        ],
        ascending=[False, True, False]
    )
    .iloc[0]
)

BEST_ALGORITHM = str(
    best_solution_row['algorithm']
)

print("Selected clustering algorithm:", BEST_ALGORITHM)
print("Configuration:", best_solution_row['configuration'])
display(best_solution_row.to_frame('value'))

# 32. Fit the Selected Solution for Profiling

- **KMeans:** can be fit on all PCA rows and can later predict new points.
- **DBSCAN:** can be expensive on very large data and does not provide normal `predict()` for new points.
- **Agglomerative:** can be expensive on large data and does not provide normal `predict()` for new points.

The notebook therefore uses the full unique-flow dataset for KMeans and a practical sample for the other algorithms when needed.


In [ ]:
if BEST_ALGORITHM == 'KMeans':

    final_cluster_model = KMeans(
        n_clusters=BEST_K,
        random_state=RANDOM_STATE,
        n_init=10
    )

    final_cluster_labels = (
        final_cluster_model
        .fit_predict(X_pca)
    )

    profile_indices = np.arange(
        len(X_pca)
    )


elif BEST_ALGORITHM == 'DBSCAN':

    FINAL_DBSCAN_MAX_ROWS = min(
        40_000,
        len(X_pca)
    )

    profile_indices = rng.choice(
        len(X_pca),
        size=FINAL_DBSCAN_MAX_ROWS,
        replace=False
    )

    final_cluster_model = DBSCAN(
        eps=BEST_DBSCAN_EPS,
        min_samples=BEST_DBSCAN_MIN_SAMPLES
    )

    final_cluster_labels = (
        final_cluster_model
        .fit_predict(
            X_pca[profile_indices]
        )
    )


else:

    FINAL_AGG_MAX_ROWS = min(
        5_000,
        len(X_pca)
    )

    profile_indices = rng.choice(
        len(X_pca),
        size=FINAL_AGG_MAX_ROWS,
        replace=False
    )

    final_cluster_model = AgglomerativeClustering(
        n_clusters=BEST_AGG_CLUSTERS,
        linkage=BEST_AGG_LINKAGE
    )

    final_cluster_labels = (
        final_cluster_model
        .fit_predict(
            X_pca[profile_indices]
        )
    )


print("Profiling rows :", len(profile_indices))
print(
    "Cluster labels :",
    np.unique(final_cluster_labels)
)

# 33. Cluster Profiling

Profiling answers:

> **What does each discovered cluster look like?**

We interpret clusters using the **original readable feature values**, not standardized PCA coordinates.

Because duplicate frequencies were preserved, we also show how many original traffic rows each cluster represents.


In [ ]:
profile_df = (
    clean_df
    .iloc[profile_indices]
    .copy()
    .reset_index(drop=True)
)

profile_df['cluster'] = final_cluster_labels

cluster_sizes = (
    profile_df
    .groupby('cluster', observed=True)
    .agg(
        unique_behavior_rows=('cluster', 'size'),
        original_rows_represented=('row_frequency', 'sum')
    )
)

cluster_sizes['original_row_percent'] = (
    cluster_sizes['original_rows_represented']
    / cluster_sizes['original_rows_represented'].sum()
    * 100
)

display(cluster_sizes)

### Analysis

In [ ]:
                                                              

preferred_profile_features = [
    'duration',
    'source_packets',
    'destination_packets',
    'source_bytes',
    'destination_bytes',
    'transmission_rate',
    'source_load',
    'destination_load',
    'source_packet_loss',
    'destination_packet_loss',
    'source_jitter',
    'destination_jitter',
    'tcp_round_trip_time',
    'source_mean_packet_size',
    'destination_mean_packet_size',
]

profile_numeric_features = [
    c for c in preferred_profile_features
    if c in profile_df.columns
]

numeric_cluster_profile = (
    profile_df
    .groupby('cluster', observed=True)[profile_numeric_features]
    .median()
)

display(numeric_cluster_profile.T)

### Results

In [ ]:
                                                                        
                                                                  

global_median = profile_df[
    profile_numeric_features
].median()

global_q1 = profile_df[
    profile_numeric_features
].quantile(0.25)

global_q3 = profile_df[
    profile_numeric_features
].quantile(0.75)

global_iqr = (
    global_q3 - global_q1
).replace(0, np.nan)

cluster_robust_deviation = (
    numeric_cluster_profile - global_median
).div(global_iqr)

for cluster_id in cluster_robust_deviation.index:

    top_features = (
        cluster_robust_deviation
        .loc[cluster_id]
        .abs()
        .sort_values(ascending=False)
        .head(5)
        .index
    )

    print(f"\nCluster {cluster_id} — strongest distinguishing features:")

    for feature in top_features:
        signed_value = (
            cluster_robust_deviation
            .loc[cluster_id, feature]
        )

        direction = (
            'higher than typical'
            if signed_value > 0
            else 'lower than typical'
        )

        print(
            f"- {feature}: {direction} "
            f"(robust deviation={signed_value:.2f})"
        )

### Results

In [ ]:
                                                    

categorical_profile_features = [
    c for c in [
        'protocol',
        'network_service',
        'connection_state'
    ]
    if c in profile_df.columns
]

for col in categorical_profile_features:

    print(f"\n{'=' * 70}")
    print(f"Weighted {col} distribution by cluster")
    print('=' * 70)

    weighted_counts = (
        profile_df
        .pivot_table(
            index='cluster',
            columns=col,
            values='row_frequency',
            aggfunc='sum',
            fill_value=0,
            observed=True
        )
    )

    weighted_percent = (
        weighted_counts
        .div(
            weighted_counts.sum(axis=1),
            axis=0
        )
        * 100
    )

                                                             
    top_categories = (
        weighted_counts
        .sum(axis=0)
        .sort_values(ascending=False)
        .head(10)
        .index
    )

    display(
        weighted_percent[
            top_categories
        ].round(2)
    )

# 34. External Check with the Provided Labels

This step happens **after clustering**.

The labels were not used to form clusters.

We now ask:

- Does a discovered cluster contain mostly normal or attack traffic?
- Do some clusters align with particular attack categories?
- How strongly do the discovered groups agree with the supplied labels?

Useful external agreement metrics:

- **ARI (Adjusted Rand Index)** — higher agreement is better
- **NMI (Normalized Mutual Information)** — higher agreement is better

These are not used as training targets.


In [ ]:
if 'attack_label' in profile_df.columns:

    weighted_label_counts = (
        profile_df
        .pivot_table(
            index='cluster',
            columns='attack_label',
            values='row_frequency',
            aggfunc='sum',
            fill_value=0,
            observed=True
        )
    )

    weighted_label_percent = (
        weighted_label_counts
        .div(
            weighted_label_counts.sum(axis=1),
            axis=0
        )
        * 100
    )

    print("Attack-label percentage inside each cluster:")
    display(weighted_label_percent.round(2))

    ari_attack = adjusted_rand_score(
        profile_df['attack_label'],
        profile_df['cluster']
    )

    nmi_attack = normalized_mutual_info_score(
        profile_df['attack_label'],
        profile_df['cluster']
    )

    print(f"ARI vs attack_label: {ari_attack:.4f}")
    print(f"NMI vs attack_label: {nmi_attack:.4f}")

### Results

In [ ]:
if 'attack_category' in profile_df.columns:

    weighted_attack_categories = (
        profile_df
        .pivot_table(
            index='cluster',
            columns='attack_category',
            values='row_frequency',
            aggfunc='sum',
            fill_value=0,
            observed=True
        )
    )

    weighted_attack_category_percent = (
        weighted_attack_categories
        .div(
            weighted_attack_categories.sum(axis=1),
            axis=0
        )
        * 100
    )

    print("Attack-category percentage inside each cluster:")
    display(
        weighted_attack_category_percent.round(2)
    )

    nmi_attack_category = normalized_mutual_info_score(
        profile_df['attack_category'].astype(str),
        profile_df['cluster']
    )

    print(
        "NMI vs attack_category:",
        f"{nmi_attack_category:.4f}"
    )

# 35. Final PCA Visualization

The clustering model may use many PCA components.

For visualization only:

- X-axis = PC1
- Y-axis = PC2
- color = discovered cluster

PC1 and PC2 are **not the only components used by the clustering model**.


In [ ]:
plot_limit = min(
    10_000,
    len(profile_indices)
)

plot_positions = rng.choice(
    len(profile_indices),
    size=plot_limit,
    replace=False
)

plot_pca_values = X_pca[
    profile_indices[plot_positions]
]

plot_labels = final_cluster_labels[
    plot_positions
]

plt.figure(figsize=(10, 7))
plt.scatter(
    plot_pca_values[:, 0],
    plot_pca_values[:, 1],
    c=plot_labels,
    s=12,
    alpha=0.55
)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(
    f'PCA Visualization — {BEST_ALGORITHM} Clusters'
)
plt.tight_layout()
plt.show()

# 36. Save the Clustering Artifacts

The saved bundle contains:

- fitted preprocessing transformer
- fitted PCA
- selected clustering model
- rename mapping
- selected feature columns
- clustering configuration
- feature-type lists

### Deployment note

`KMeans` supports assigning new points with `.predict()`.

Standard DBSCAN and Agglomerative Clustering do not provide the same direct prediction workflow for unseen samples, so a new-point Streamlit app is generated only when KMeans is the selected final solution.


In [ ]:
MODEL_BUNDLE_PATH = 'unsw_nb15_clustering_bundle.joblib'

model_bundle = {
    'rename_map': RENAME_MAP,
    'selected_feature_columns': selected_feature_cols,
    'feature_drop_columns': feature_drop_cols,
    'categorical_columns': categorical_feature_cols,
    'binary_columns': binary_feature_cols,
    'skewed_numeric_columns': skewed_numeric_cols,
    'regular_numeric_columns': regular_numeric_cols,
    'preprocessor': preprocessor,
    'pca': pca,
    'cluster_model': final_cluster_model,
    'best_algorithm': BEST_ALGORITHM,
    'best_kmeans_k': BEST_K,
    'best_dbscan_eps': BEST_DBSCAN_EPS,
    'best_dbscan_min_samples': BEST_DBSCAN_MIN_SAMPLES,
    'best_agglomerative_clusters': BEST_AGG_CLUSTERS,
    'best_agglomerative_linkage': BEST_AGG_LINKAGE,
}

joblib.dump(
    model_bundle,
    MODEL_BUNDLE_PATH
)

print(
    "Saved clustering bundle to:",
    MODEL_BUNDLE_PATH
)

### Export Results

In [ ]:
                                                                    

CLUSTERED_OUTPUT_PATH = 'unsw_nb15_clustered_profiles.csv'

profile_df.to_csv(
    CLUSTERED_OUTPUT_PATH,
    index=False
)

print(
    "Saved clustered profile data to:",
    CLUSTERED_OUTPUT_PATH
)

# 37. Streamlit Deployment Template

If KMeans wins, the app can accept a batch CSV/Parquet file and assign each new network flow to a discovered cluster.

The user uploads raw UNSW-style data; the app applies the same rename → preprocessing → PCA → KMeans pipeline.


In [ ]:
if BEST_ALGORITHM == 'KMeans':

    app_code = r"""
import joblib
import numpy as np
import pandas as pd
import streamlit as st

BUNDLE_PATH = 'unsw_nb15_clustering_bundle.joblib'

bundle = joblib.load(BUNDLE_PATH)

preprocessor = bundle['preprocessor']
pca = bundle['pca']
model = bundle['cluster_model']
rename_map = bundle['rename_map']
required_features = bundle['selected_feature_columns']

st.set_page_config(
    page_title='Network Traffic Clustering',
    page_icon='🌐'
)

st.title('🌐 UNSW-NB15 Network Traffic Clustering')
st.write(
    'Upload network-flow data to assign each row '
    'to the learned KMeans behavior cluster.'
)

uploaded = st.file_uploader(
    'Upload CSV or Parquet',
    type=['csv', 'parquet']
)

if uploaded is not None:

    if uploaded.name.lower().endswith('.csv'):
        new_df = pd.read_csv(uploaded)
    else:
        new_df = pd.read_parquet(uploaded)

    new_df = new_df.rename(
        columns=rename_map
    )

    missing_features = [
        c for c in required_features
        if c not in new_df.columns
    ]

    if missing_features:
        st.error(
            'Missing required features: '
            + ', '.join(missing_features)
        )
    else:
        X_new = new_df[
            required_features
        ].copy()

        X_processed = preprocessor.transform(
            X_new
        )

        X_pca_new = pca.transform(
            X_processed
        )

        clusters = model.predict(
            X_pca_new
        )

        result = new_df.copy()
        result['cluster'] = clusters

        st.subheader('Cluster counts')
        st.dataframe(
            result['cluster']
            .value_counts()
            .sort_index()
            .rename('count')
        )

        st.subheader('Clustered rows')
        st.dataframe(result.head(500))

        if X_pca_new.shape[1] >= 2:
            chart_df = pd.DataFrame({
                'PC1': X_pca_new[:, 0],
                'PC2': X_pca_new[:, 1],
                'cluster': clusters.astype(str)
            })

            st.subheader('PCA view')
            st.scatter_chart(
                chart_df,
                x='PC1',
                y='PC2',
                color='cluster'
            )
"""

    with open(
        'app.py',
        'w',
        encoding='utf-8'
    ) as f:
        f.write(app_code)

    print("Created app.py")
    print("Run with: streamlit run app.py")

else:
    print(
        f"Final algorithm is {BEST_ALGORITHM}. "
        "A direct new-point prediction Streamlit template "
        "was not generated because standard "
        f"{BEST_ALGORITHM} does not provide KMeans-like predict()."
    )

### Results

In [ ]:
requirements = '''pandas
numpy
scikit-learn
joblib
streamlit
pyarrow
'''

with open(
    'requirements.txt',
    'w',
    encoding='utf-8'
) as f:
    f.write(requirements)

print("Created requirements.txt")